# 06. ALTER · DROP · TRUNCATE

*Module: 2 — Data definition · Companion summary: `lectures/06-alter-drop-and-truncate.md` · SQL engine: SQLite (in-memory)*

## What you will study

- add and remove columns with ALTER TABLE
- know why changing a populated column's datatype can fail
- choose between DELETE-rows, TRUNCATE and DROP TABLE

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Start from the author table (course story arc)

In [ ]:
q("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT NOT NULL, firstname TEXT NOT NULL, email TEXT, city TEXT, country TEXT)")
q("INSERT INTO author VALUES ('A1', 'Chong', 'Raul', 'RFC@ca.ibm.com', 'Toronto', 'CA')")
q("SELECT * FROM author")

## 🧪 ALTER: add a telephone number column

In [ ]:
q("ALTER TABLE author ADD COLUMN telephone_number TEXT")   # strings, not numbers!
q("UPDATE author SET telephone_number = '+1 (416) 555-0100' WHERE author_id = 'A1'")
q("SELECT * FROM author")

**📕 Dialect note — changing a column's datatype.** The course shows
```sql
ALTER TABLE author MODIFY telephone_number CHAR(20);     -- DB2 / MySQL
ALTER TABLE author ALTER COLUMN telephone_number CHAR(20); -- SQL Server syntax
```
SQLite **cannot change a column's type at all** — the standard recipe is:
create the corrected table, `INSERT INTO new SELECT … FROM old`, drop old, rename new:

In [ ]:
q("CREATE TABLE author_v2 (author_id TEXT PRIMARY KEY, lastname TEXT NOT NULL, firstname TEXT NOT NULL, email TEXT, city TEXT, country TEXT, telephone_number TEXT)")
q("INSERT INTO author_v2 SELECT author_id, lastname, firstname, email, city, country, telephone_number FROM author")
q("DROP TABLE author")
q("ALTER TABLE author_v2 RENAME TO author")
q("SELECT * FROM author")

## 🧪 ALTER again: drop the column when the spec changes

In [ ]:
q("ALTER TABLE author DROP COLUMN telephone_number")   # SQLite ≥ 3.35 supports this
q("SELECT * FROM author")

## 🧪 DROP TABLE vs TRUNCATE/DELETE

In [ ]:
q("CREATE TABLE scratch (x INTEGER)")
q("INSERT INTO scratch VALUES (1),(2)")
q("DELETE FROM scratch")   # SQLite's TRUNCATE equivalent
q("SELECT count(*) AS rows_left FROM scratch")   # table survives, 0 rows
q("DROP TABLE scratch")   # DDL: container gone


## 💡 Tips & tricks

- Store phone numbers as **strings** — `( ) + -` break numeric columns.
- Mnemonic: TRUNCATE = fast *empty*; DROP = *demolish*. Neither takes a WHERE.

## 🎯 Your turn

1. `ALTER TABLE author ADD COLUMN nickname TEXT`, populate it, then drop the column again.
2. Build `tmp1`, empty it with DELETE, and prove the empty table still exists.

In [ ]:
# 💡 Solution
q("ALTER TABLE author ADD COLUMN nickname TEXT")
q("UPDATE author SET nickname = 'RC' WHERE author_id = 'A1'")
q("SELECT author_id, nickname FROM author")
q("ALTER TABLE author DROP COLUMN nickname")
q("SELECT * FROM author")
q("CREATE TABLE tmp1 (x INTEGER)")
q("INSERT INTO tmp1 VALUES (9)")
q("DELETE FROM tmp1")
q("SELECT count(*) AS n FROM tmp1")